In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


results = pd.read_csv('../outputs/forecast_results.csv', parse_dates=['date'])


results['error'] = results['demand'] - results['predicted_demand']
results['abs_error'] = results['error'].abs()

print(results.head(10))
print(f"\nAverage absolute error: {results['abs_error'].mean():.2f}")
print(f"Max error: {results['abs_error'].max():.2f}")
print(f"Min error: {results['abs_error'].min():.2f}")

        date  demand  predicted_demand  error  abs_error
0 2023-01-01   135.0            126.14   8.86       8.86
1 2023-01-02   124.0            130.36  -6.36       6.36
2 2023-01-03   138.0            130.12   7.88       7.88
3 2023-01-04   118.0            129.19 -11.19      11.19
4 2023-01-05   145.0            128.58  16.42      16.42
5 2023-01-06   140.0            128.82  11.18      11.18
6 2023-01-07   138.0            129.53   8.47       8.47
7 2023-01-08   146.0            130.36  15.64      15.64
8 2023-01-09   153.0            133.17  19.83      19.83
9 2023-01-10   147.0            137.96   9.04       9.04

Average absolute error: 14.36
Max error: 45.50
Min error: 0.01


In [ ]:

mean_demand = results['demand'].mean()
std_demand = results['demand'].std()
mean_error = results['abs_error'].mean()
std_error = results['error'].std()


cv = std_demand / mean_demand


bias = results['error'].mean()


var_95 = np.percentile(results['abs_error'], 95)

print("=== Demand Risk Report ===")
print(f"Mean Demand:          {mean_demand:.2f} units")
print(f"Std of Demand:        {std_demand:.2f} units")
print(f"Coefficient of Variation (CV): {cv:.4f} ({cv*100:.2f}%)")
print(f"Mean Forecast Error:  {mean_error:.2f} units")
print(f"Forecast Bias:        {bias:.2f} units")
print(f"95th Percentile Error (VaR): {var_95:.2f} units")

=== Demand Risk Report ===
Mean Demand:          142.95 units
Std of Demand:        14.70 units
Coefficient of Variation (CV): 0.1028 (10.28%)
Mean Forecast Error:  14.36 units
Forecast Bias:        12.23 units
95th Percentile Error (VaR): 31.99 units


In [ ]:
def classify_risk(cv):
    """
    Translate CV into a risk level the 
    financing engine can use
    """
    if cv < 0.10:
        return 'Low Risk'
    elif cv < 0.20:
        return 'Medium Risk'
    else:
        return 'High Risk'

def calculate_risk_adjusted_demand(mean_demand, std_demand, risk_level):
    """
    Adjust demand forecast based on risk level.
    Higher risk = more conservative (lower) demand estimate
    """
    if risk_level == 'Low Risk':
        
        return mean_demand + 0.5 * std_demand
    elif risk_level == 'Medium Risk':
        
        return mean_demand
    else:
        
        return mean_demand - 0.5 * std_demand


risk_level = classify_risk(cv)
risk_adjusted_demand = calculate_risk_adjusted_demand(
    mean_demand, std_demand, risk_level
)

print(f"Demand Volatility (CV):     {cv*100:.2f}%")
print(f"Risk Level:                 {risk_level}")
print(f"Raw Mean Demand:            {mean_demand:.2f} units")
print(f"Risk Adjusted Demand:       {risk_adjusted_demand:.2f} units")
print(f"Adjustment:                 {risk_adjusted_demand - mean_demand:+.2f} units")

Demand Volatility (CV):     10.28%
Risk Level:                 Medium Risk
Raw Mean Demand:            142.95 units
Risk Adjusted Demand:       142.95 units
Adjustment:                 +0.00 units


In [ ]:

risk_summary = {
    'mean_demand': mean_demand,
    'std_demand': std_demand,
    'cv': cv,
    'risk_level': risk_level,
    'risk_adjusted_demand': risk_adjusted_demand,
    'forecast_bias': bias,
    'var_95': var_95,
    'mean_forecast_error': mean_error
}


risk_df = pd.DataFrame([risk_summary])
risk_df.to_csv('../outputs/risk_summary.csv', index=False)

print("=== Risk Summary Saved ===")
for key, value in risk_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

=== Risk Summary Saved ===
mean_demand: 142.9534
std_demand: 14.7013
cv: 0.1028
risk_level: Medium Risk
risk_adjusted_demand: 142.9534
forecast_bias: 12.2299
var_95: 31.9920
mean_forecast_error: 14.3568
